<a href="https://colab.research.google.com/github/isaacadebayo/Agentic_projects/blob/main/multimodal_Chatbot_image_text_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install once restart session and comment out

In [ ]:
#!pip install mlflow cryptography==48.0.0 --ignore-installed blinker

  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cffi-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.6 kB)
  Using cached mlflow_skinny-3.14.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.14.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached gunicorn-26.0.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached huey-3.0.3-py3-none-any.whl.metadata (4.5 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-ma

In [ ]:
#pip install --upgrade cryptography==48.0.0

In [2]:
%%time
! pip install --upgrade gradio
!pip install -q langchain langchain-openai faiss-cpu pypdf
!pip install langchain langchain-community
!pip install langchain langchain-huggingface
! pip install bs4
! pip install langchain
! pip install langchain-core
! pip install openai
! pip install -q google-genai openai
! pip install -q langgraph google-genai
! pip install sentence-transformers
! pip install edge-tts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 whi

If installing mlflow --cryptography causes conflict in package. Need to force reinstall libary and package, end session and re-run

In [ ]:
# Aggressively uninstall all potentially conflicting packages
'''!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-huggingface pydantic mlflow gradio pyngrok cryptography

# Install a recent and stable Langchain version, which should pull in compatible langchain-core etc.
!pip install --upgrade --force-reinstall -q langchain==0.2.11

# Explicitly install a stable Pydantic v2 version that is known to work with Langchain 0.2.x
# Using --force-reinstall to ensure it overrides any Pydantic installed by Langchain if needed
!pip install --upgrade --force-reinstall -q pydantic==2.7.1

# Install other core Langchain components, ensuring they are also recent and compatible
!pip install --upgrade -q langchain-community==0.2.11 langchain-openai==0.1.15 langchain-huggingface==0.0.3

# Install remaining dependencies
!pip install --upgrade -q faiss-cpu pypdf bs4 openai google-genai langgraph sentence-transformers edge-tts mlflow pyngrok cryptography==43.0.3 gradio'''

In [3]:
%%time
import gradio as gr
import getpass
import os
import bs4
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing_extensions import List, TypedDict
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from google import genai
from google.genai import types
from google.colab import drive
import edge_tts

<timed exec>:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: user 25.6 s, sys: 3.35 s, total: 29 s
Wall time: 35.3 s


In [4]:
from google.colab import drive
drive.mount('/content/drive')

dataset = '/content/drive/MyDrive/corpus_first_100.csv'

Mounted at /content/drive


In [5]:
import pandas as pd
from langchain_core.documents import Document # Import Document class

loader_df = pd.read_csv(dataset)

# Convert DataFrame rows to Document objects
docs = []
for index, row in loader_df.iterrows():
    docs.append(Document(page_content=row['text']))

# Join the page content from the loaded documents into a single string for test_chatbot
financial_chatbot = "\n\n".join([doc.page_content for doc in docs])

In [ ]:
financial_chatbot

In [6]:
# 2. Setting up Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup complete. Your chatbot is ready for testing, Let's Go Isaac!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup complete. Your chatbot is ready for testing, Let's Go Isaac!


In [7]:
from google.colab import userdata

In [8]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [34]:
%%time
llm = ChatOpenAI(model="gpt-4o", temperature=0.5, max_tokens=250)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

CPU times: user 98.8 ms, sys: 1.24 ms, total: 100 ms
Wall time: 92.3 ms


[Trace(trace_id=tr-5253e24a1a6fb8d49e81fbb85905870a), Trace(trace_id=tr-a321d26c658a211c70e15d49bddd775e), Trace(trace_id=tr-8ba562764237c87abd38b737b5d4d933), Trace(trace_id=tr-297859e95035b14b6832b0c8f7a7a96f), Trace(trace_id=tr-d0e05a3d1b273d68519dc5252c073ed9)]

In [35]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [36]:
%%time
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
all_splits = text_splitter.split_documents(docs)

CPU times: user 33.5 ms, sys: 0 ns, total: 33.5 ms
Wall time: 34.1 ms


In [37]:
%%time
# Create FAISS index from chunks
db = FAISS.from_documents(all_splits, embeddings)

CPU times: user 21.7 s, sys: 1.21 s, total: 22.9 s
Wall time: 28.5 s


Trace(trace_id=tr-48d30b33fe4c858fcb9c1455a4612478)

In [13]:
import openai
import os # Ensure os is imported for environment variable access
import asyncio # Added for async TTS
import edge_tts # Added for TTS
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chatbot_prompt = ChatPromptTemplate.from_messages([
     ("system", """You are a financial analyst at a stock firm. You need to answer questions related to some financial review in a  polite manner to a prospective client.\n
      Context:\n# {context}"""),
     ("human", "Question: {question}"),
     # Removed: ("ai", "Answer: "), to prevent incomplete AI responses
])

# New prompt for rephrasing the question using chat history
condense_question_prompt = ChatPromptTemplate.from_messages([
     ("system", "Given a chat history and a follow-up question, rephrase the follow-up question to be a standalone question."),
     MessagesPlaceholder(variable_name="chat_history"),
     ("human", "{question}"),
])

def ask_question(question: str, history=None) -> str:
    chat_history_for_llm = []

    if history:
        for turn in history:
            # Safely handle dictionary format, Gradio message objects, or legacy tuples
            if isinstance(turn, dict):
                role = turn.get("role")
                content = turn.get("content", "")
                if role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=content))
                elif role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=content))
            elif hasattr(turn, "role"):  # If it arrives as a Gradio ChatMessage object
                if turn.role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=turn.content))
                elif turn.role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=turn.content))
            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                if turn[0]: chat_history_for_llm.append(HumanMessage(content=turn[0]))
                if turn[1]: chat_history_for_llm.append(AIMessage(content=turn[1]))

    standalone_question = question
    if chat_history_for_llm:
        try:
            # Condense follow-up question using chat history
            condense_chain = condense_question_prompt | llm
            standalone_question = condense_chain.invoke({
                "question": question,
                "chat_history": chat_history_for_llm
            }).content
        except Exception as e:
            print(f"Error condensing question: {e}")
            standalone_question = question

    # Retrieve matching vector documents
    try:
        retrieved_docs = db.similarity_search(standalone_question, k=4)
    except NameError:
        retrieved_docs = vector_store.similarity_search(standalone_question, k=4)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs if hasattr(doc, 'page_content'))

    # Format the prompt messages
    messages = chatbot_prompt.format_messages(question=standalone_question, context=docs_content)

    # Convert LangChain message formats to raw dictionary maps to prevent SDK validation rejections
    formatted_payload = []
    for msg in messages:
        # Standardize roles into API compatible tags
        role_type = "user" if msg.type == "human" else "system" if msg.type == "system" else "assistant"
        formatted_payload.append({"role": role_type, "content": msg.content})

    # Invoke your model using the safe dictionary array payload
    return llm.invoke(formatted_payload).content


def predict(input, history):
     # This 'predict' function is not used by the multimodal_app or agent_predict, but leaving it as is.
     output = ask_question(input, history)

     history.append((input, output))

     return history, history


# This converts voice to text using OpenAI Whisper
def transcribe_audio(audio_path):
    # Ensure OPENAI_API_KEY is set in environment or Colab secrets
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY environment variable not set."

    client = openai.OpenAI()
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text

# Added text-to-speech function
async def _tts(text):
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response.mp3"
    await communicate.save(path)
    return path


# This function will handle both text and transcribed voice input
# and manage the chat history for gr.Chatbot. It uses the globally defined ask_question.
# It now expects and returns history in the Gradio Chatbot format (list of lists/tuples).
def predict_for_multimodal(message, history):
    if history is None:
        history = []

    # 1. Compute the text generation answer
    rag_answer = ask_question(message, history)

    # 2. Append history using explicit dictionary format definitions to guarantee Gradio matching
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": rag_answer})

    return history, ""

# This function wraps transcribe_audio and then calls the main predict_for_multimodal and includes TTS
def voice_chat_handler(audio_path, history):
    status_msg = "" # For the status_box
    audio_output_path = None
    user_text = "" # Initialize user_text
    bot_text = "" # Initialize bot_text

    if history is None:
        history = []

    if audio_path is None:
        status_msg = "No audio input received. Please record your question."
        return history, audio_output_path, status_msg

    try:
        # 1. Transcribe audio
        status_msg = "Converting audio to text..."
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            status_msg = user_text
            history.append({"role": "assistant", "content": status_msg})
            return history, audio_output_path, status_msg

        # 2. Get RAG answer
        status_msg = f"User: {user_text}\nGenerating response..."
        bot_text = ask_question(user_text, history) # Use ask_question here

        # 3. Convert bot's response to audio
        status_msg = "Converting response to speech..."
        audio_output_path = asyncio.run(_tts(bot_text))

        status_msg = "Response generated successfully!"

    except Exception as e:
        status_msg = f"Error in voice processing: {e}"
        bot_text = f"I encountered an error while processing your request: {e}"
        audio_output_path = None

    # 4. Update history
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": bot_text})

    return history, audio_output_path, status_msg


with gr.Blocks() as multimodal_app:
    gr.Markdown("## Multimodal RAG Assistant")

    chatbot = gr.Chatbot(label="Chat History")
    # Added status_box and audio_out components
    status_box = gr.Textbox(label="Voice Status", interactive=False, visible=True)
    audio_out = gr.Audio(label="🔊 Voice Response", autoplay=True, streaming=True) # Added for TTS output

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a question...", scale=4)
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1)

    msg.submit(
        predict_for_multimodal,
        [msg, chatbot],
        [chatbot, msg] # Clear the text input after submission
    )

    audio_in.stop_recording(
        voice_chat_handler,
        [audio_in, chatbot],
        [chatbot, audio_out, status_box] # Updated outputs for voice_chat_handler
    )

multimodal_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf20dca8d027e65c4f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Asynccio is a library used to write conccurreent code using the async/await syntax which allow program to peerform multiple operations at the same time without using multiple threads or processes.

Asyncio event loop watcches and eevent and dispatch them to the approrpaiate asyncio task. If a task need data from aa network it awaits that operation and the event loop swtich to another task

#Multimodal with image generation

Can generate an image on the first prompt but when RAG is invoked to retrieve financial information from my dataset. Image generation stops

In [18]:
!pip install dvc[gdrive]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1

### Connecting GitHub

In [19]:
# Replace with your GitHub email and username
!git config --global user.email "isaac_adebayojr@gmail.com"
!git config --global user.name "isaacadebayo"

print("Git user configured.")

Git user configured.


In [20]:
# Store your GitHub PAT securely in Colab Secrets
# Click on the '🔑' icon on the left panel, add a new secret named 'GH_TOKEN' and paste your PAT there.
from google.colab import userdata
import os

# Get the PAT from Colab secrets
GH_TOKEN = userdata.get('GH_TOKEN')

# Set it as an environment variable (optional, but good practice if cloning private repos)
os.environ['GH_TOKEN'] = GH_TOKEN

print("GitHub Token retrieved. You can now use it for cloning private repositories.")

GitHub Token retrieved. You can now use it for cloning private repositories.


In [21]:
# Replace 'your_username' and 'your_repository' with your actual GitHub details.
# For a public repository:
!git clone https://github.com/isaacadebayo/Agentic_projects.git

# For a private repository using PAT:
# This uses the GH_TOKEN environment variable
!git clone https://{GH_TOKEN}@github.com/isaacadebayo/Agentic_projects

# Change into your repository directory to perform Git operations
%cd Agentic_projects

print("Repository cloned. You are now in the repository directory.")

Cloning into 'Agentic_projects'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 92 (delta 49), reused 16 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 933.54 KiB | 8.12 MiB/s, done.
Resolving deltas: 100% (49/49), done.
fatal: destination path 'Agentic_projects' already exists and is not an empty directory.
/content/Agentic_projects
Repository cloned. You are now in the repository directory.


In [22]:
# Authenticate Google Cloud for DVC to access GCS
# This will open a browser window for authentication.
# Follow the instructions to get an authorization code and paste it back.
!gcloud auth application-default login

print("Google Cloud authentication complete. You can now retry the DVC push.")

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=UE548eCkg5Na8c9GIghTx5YwqMqjEG&prompt=consent&token_usage=remote&access_type=offline&code_challenge=ySwhF_DFmeszxO2fwnnuO5N0m3owxkRwY8XY3ZaBJaI&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AdkVLPw7kPfpjGK1IgaPhYq-8BfomsFrtWJEP4b0YOxgiUKnm4AovVeA2H-qhj6FJZbz3g

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [23]:
# Initialize DVC within the current directory (which is the cloned Git repo)
!dvc init

# Configure Google Drive or cloud storage as a remote storage (replace 'dvc_storage' with your desired folder name in Google Drive)
!dvc remote add -d gdrive_remote gdrive://MyDrive/dvc_storage
#!dvc remote add -d gcs_remote gs://my-agentic-chatbot-data-lake

# Create a data directory within the Git repository if it doesn't exist
!mkdir -p data

# Copy the data file into the Git repository's data directory
!cp /content/drive/MyDrive/corpus_first_100.csv data/corpus_first_100.csv

# Add a data file to DVC. This will create a .dvc file at data/longevity.pdf.dvc
!dvc add data/corpus_first_100.csv

# Ensure mlflow.db and proxy.py are ignored by Git
# Append to the main .gitignore in the repo root, creating it if it doesn't exist
# Check if .gitignore exists, create if not, then append
gitignore_path = ".gitignore"
with open(gitignore_path, "a+") as f:
    f.seek(0)
    existing_content = f.read()
    if "mlflow.db" not in existing_content:
        f.write("\nmlflow.db\n")
    if "proxy.py" not in existing_content:
        f.write("proxy.py\n")

# Stage DVC file and any modified .gitignore files (both root and data folder)
!git add data/corpus_first_100.csv.dvc
!git add .gitignore  # Stage the root .gitignore
!git add data/.gitignore # Stage data/.gitignore which DVC might have modified

# Commit the staged changes
!git commit -m "Add corpus_first_100.csv DVC file and update gitignore"

# Push the DVC-tracked data to the remote (Google Drive / GCS)
!dvc push

print("DVC setup complete. Your data file is now versioned with DVC and stored in Google Drive.")
print("You can push/pull changes using: !dvc push and !dvc pull")

ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.
Setting 'gdrive_remote' as a default remote.
ERROR: configuration error - config file error: remote 'gdrive_remote' already exists. Use `-f|--force` to overwrite it.
⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
ERROR:  output 'data/corpus_first_100.csv' is already tracked by SCM (e.g. Git).
    You can remove it from Git, then add to DVC.
        To stop tracking from Git:
            git rm -r --cached 'data/corpus_first_100.csv'
            git commit -m "stop tracking data/corpus_first_100.csv" 
fatal: pathspec 'data/corpus_first_100.csv.dvc' did not match any files
fatal: pathspec 'data/.gitignore' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Pushing
Everything is up to date.
DVC setup complete. Your data file is now versioned w

In [24]:
# Stage all changes in the current directory
!git add .

# Commit the staged changes with a message
# Replace 'Your insightful commit message' with a description of your changes
!git commit -m "Testing commit from colab to github"

# Push the changes to the 'main' branch of your remote repository
# This uses the GH_TOKEN environment variable for authentication
!git push https://{GH_TOKEN}@github.com/isaacadebayo/Agentic_projects.git main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [25]:
print('Contents of the cloned repository:')
!ls -F

Contents of the cloned repository:
Agentic_chatbot_mcp_client_server_GCS.ipynb
Agentic_chatbot_mcp_client_server_multiple_files_type.ipynb
Agentic_multimodal_Chatbot.ipynb
Apple_Stock_prediction_COMPLETE.ipynb
data/
GRADIO_Multimodal_Chatbot_Longevity_COMPLETE_.ipynb
GRADIO_Multimodal_Chatbot_Longevity_MLflow.ipynb
multimodal_Chatbot_image_text_audio.ipynb
README.md


In [26]:
!pip install pyngrok

In [15]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.

Need pip install mlflow for this to run

# MLFlow

In [27]:
import mlflow
import mlflow.langchain
import mlflow.openai
# Define a wrapper function to integrate MLflow tracking for each chatbot interaction
def mlflow_tracked_ask_question(question: str, history=None) -> str:
    # Start an MLflow run for each interaction
    with mlflow.start_run(run_name=f"Query: {question[:50]}...") as run:
        mlflow.log_param("user_question", question)

        # Log chat history as a parameter or artifact (as JSON string)
        history_json = json.dumps(history if history is not None else [], indent=2)
        mlflow.log_text(history_json, "chat_history_before_query.json")
        mlflow.log_param("chat_history_length", len(history) if history else 0)

        # Log prompt templates as artifacts
        mlflow.log_text(chatbot_prompt.format(question="<QUESTION_PLACEHOLDER>", context="<CONTEXT_PLACEHOLDER>"), "chatbot_prompt_template.txt")
        mlflow.log_text(condense_question_prompt.format(question="<QUESTION_PLACEHOLDER>", chat_history=[]), "condense_question_prompt_template.txt")

        # Log LLM and Embedding model details
        mlflow.log_param("llm_model_name", llm.model_name) # Assuming llm is a ChatOpenAI instance
        mlflow.log_param("llm_temperature", llm.temperature)
        mlflow.log_param("llm_max_tokens", llm.max_tokens)
        mlflow.log_param("embedding_model_name", embeddings.model) # Assuming embeddings is an OpenAIEmbeddings instance
        mlflow.log_param("vector_store_type", "FAISS")

        # Call the original ask_question function to get the RAG answer
        rag_answer = ask_question(question, history)

        mlflow.log_param("bot_response", rag_answer)

        # Log input/output character counts as a proxy for token usage
        mlflow.log_metric("input_char_count", len(question))
        mlflow.log_metric("output_char_count", len(rag_answer))

        # Log response audio if available (only for voice_chat_handler)
        # Note: 'response.mp3' might not exist for text queries or if TTS fails
        try:
            if os.path.exists("response.mp3"):
                mlflow.log_artifact("response.mp3", "bot_audio_response")
        except Exception as e:
            print(f"Warning: Could not log response.mp3: {e}")

        return rag_answer

print("MLflow wrapper function 'mlflow_tracked_ask_question' defined.")

MLflow wrapper function 'mlflow_tracked_ask_question' defined.


#Ngrok

In [28]:
import subprocess, time, requests, os, re
from pyngrok import ngrok
from google.colab import userdata

# ── 1. Kill everything ────────────────────────────────────────────
subprocess.run("pkill -f 'mlflow server'", shell=True)
subprocess.run("pkill -f proxy.py", shell=True)
subprocess.run("pkill -f cloudflared", shell=True)
time.sleep(3)

# ── 2. Start MLflow ───────────────────────────────────────────────
env = os.environ.copy()
env["GUNICORN_CMD_ARGS"] = "--forwarded-allow-ips='*'"

subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:////content/Agentic_projects/mlflow.db",
     "--serve-artifacts",
     "--host", "0.0.0.0",
     "--port", "5000",
     "--cors-allowed-origins", "https://ia-mlflow-dashboard.ngrok.app"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    cwd="/content/Agentic_projects",
    env=env
)

print("Waiting for MLflow...")
for i in range(30):
    try:
        r = requests.get("http://localhost:5000/health", timeout=3)
        if r.status_code == 200:
            print(f"MLflow ready")
            break
    except Exception:
        pass
    time.sleep(2)

# ── 3. Write proxy (rewrites Host header to localhost:5000) ────────
proxy_code = """
import http.server
import http.client
import urllib.parse

class Proxy(http.server.BaseHTTPRequestHandler):
    def _proxy(self, body=None):
        parsed = urllib.parse.urlparse(f"http://localhost:5000{self.path}")

        conn = http.client.HTTPConnection("localhost", 5000, timeout=30)

        headers = dict(self.headers)
        headers["Host"] = "localhost:5000"
        headers["X-Forwarded-Host"] = "localhost:5000"
        headers["X-Forwarded-Proto"] = "http"
        headers.pop("Transfer-Encoding", None)
        headers.pop("transfer-encoding", None)

        try:
            conn.request(self.command, parsed.path + (f"?{parsed.query}" if parsed.query else ""), body=body, headers=headers)
            resp = conn.getresponse()

            self.send_response(resp.status)

            for k, v in resp.getheaders():
                if k.lower() not in ("transfer-encoding", "connection", "keep-alive"):
                    self.send_header(k, v)
            self.send_header("Access-Control-Allow-Origin", "*")
            self.send_header("Access-Control-Allow-Headers", "*")
            self.end_headers()

            # Stream response in chunks — fixes chart data loading
            while True:
                chunk = resp.read(8192)
                if not chunk:
                    break
                self.wfile.write(chunk)
            self.wfile.flush()

        except Exception as e:
            self.send_response(502)
            self.end_headers()
            self.wfile.write(str(e).encode())
        finally:
            conn.close()

    def do_GET(self):     self._proxy()
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length) if length else None)
    def do_PUT(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length) if length else None)
    def do_DELETE(self):  self._proxy()
    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, PUT, DELETE, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.end_headers()
    def log_message(self, *args): pass

server = http.server.HTTPServer(('0.0.0.0', 5001), Proxy)
server.serve_forever()
"""

with open("proxy.py", "w") as f:
    f.write(proxy_code)

subprocess.Popen(["python", "proxy.py"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Proxy started on port 5001")
time.sleep(3)

# ── 4. Connect ngrok to proxy on port 5001 ────────────────────────
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))

# Replace with your custom domain
public_url = ngrok.connect(5001, domain="ia-mlflow-dashboard.ngrok.app")
print(f"\nMLflow UI: {public_url}\n")

Waiting for MLflow...
MLflow ready
Proxy started on port 5001

MLflow UI: NgrokTunnel: "https://ia-mlflow-dashboard.ngrok.app" -> "http://localhost:5001"



In [29]:
import mlflow
import mlflow.langchain
import mlflow.openai
import os
import openai
import json

# Set the MLflow tracking URI to connect to the server started by ngrok
# The ngrok setup usually exposes localhost:5000
mlflow.set_tracking_uri("http://localhost:5000")

# Set an experiment name for better organization in the MLflow UI
mlflow.set_experiment("Multimodal RAG Chatbot Interactions")
mlflow.openai.autolog()

print("MLflow tracking setup complete.")

2026/06/22 17:23:09 INFO mlflow.tracking.fluent: Experiment with name 'Multimodal RAG Chatbot Interactions' does not exist. Creating a new experiment.


MLflow tracking setup complete.


### Audio, text and upload image/ capture image via webcam

In [38]:
import openai
import os
import asyncio
import edge_tts
import base64
import cv2
import numpy as np # Import numpy for image handling
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# ─────────────────────────────────────────────────────
# Assumed defined upstream in your notebook: llm, db, client, gr, ChatPromptTemplate
# ─────────────────────────────────────────────────────

# ── Prompts ───────────────────────────────────────────────────────────────────

chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a financial analyst at a stock firm. "
        "Answer questions related to the financial review politely for a prospective client.\n\n"
        "You can also generate images to visualize concepts if explicitly asked. "
        "Use the `generate_dalle_image` tool when the user explicitly asks to create or generate an image.\n\n"
        "Context:\n{context}"
    )),
    ("human", "Question: {question}"),
])

condense_question_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and a follow-up question, rephrase the follow-up question to be a standalone question."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# ── Tool definitions ──────────────────────────────────────────────────────────

tools = [
    {
        "type": "function",
        "function": {
            "name": "generate_dalle_image",
            "description": (
                "Generate an image using gpt-image-1 based on a text prompt. "
                "Use this when the user explicitly asks to create or generate an image."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "visual_prompt": {
                        "type": "string",
                        "description": "A detailed text description to guide image generation.",
                    }
                },
                "required": ["visual_prompt"],
            },
        },
    },
]

# ── Core helpers ──────────────────────────────────────────────────────────────

def _build_langchain_history(history: list) -> list:
    """
    Convert Gradio history to LangChain message objects.
    Handles both dict-format {"role": ..., "content": ...}
    and tuple/list-format [user_msg, bot_msg] defensively.
    Base64 image strings are stripped to avoid token bloat.
    """
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = turn.get("content", "")
            # Skip image replies — base64 data URIs and markdown images
            if isinstance(content, str) and (content.startswith("<img") or content.startswith("![")): # type: ignore
                continue
            if role in ("user", "human") and content:
                messages.append(HumanMessage(content=content)) # type: ignore
            elif role in ("assistant", "ai", "model") and content:
                messages.append(AIMessage(content=content)) # type: ignore
        # Fallback: handle legacy tuple/list format Gradio may pass in some versions
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_content = str(turn[0]) if turn[0] else ""
            bot_content  = str(turn[1]) if turn[1] else ""
            if user_content:
                messages.append(HumanMessage(content=user_content)) # type: ignore
            # Strip image replies from tuple history too
            if bot_content and not bot_content.startswith("<img") and not bot_content.startswith("!["): # type: ignore
                messages.append(AIMessage(content=bot_content)) # type: ignore
    return messages


def generate_dalle_image(visual_prompt: str) -> str | None:
    """
    Generate an image via gpt-image-1.
    Returns a base64 data URI (or URL as fallback), or raises on failure.
    """
    print(f"[gpt-image-1] Generating image for: {visual_prompt}")
    try:
        response = client.images.generate(
            model="gpt-image-1",
            prompt=visual_prompt,
            size="1024x1024",
            n=1,
        )

        if not response or not response.data:
            raise ValueError("gpt-image-1 returned an empty response.")

        image_data = response.data[0]

        # gpt-image-1 returns base64, not a URL
        if hasattr(image_data, "b64_json") and image_data.b64_json:
            print("[gpt-image-1] Got base64 image data.")
            return f"data:image/png;base64,{image_data.b64_json}"

        # Fallback: try URL in case behaviour changes
        elif hasattr(image_data, "url") and image_data.url:
            print(f"[gpt-image-1] Got URL: {image_data.url}")
            return image_data.url

        else:
            raise ValueError("Response contained neither b64_json nor url.")

    except Exception as e:
        print(f"[gpt-image-1] Error: {e}")
        raise e


# ── RAG pipeline ──────────────────────────────────────────────────────────────

def ask_question(question: str, history: list | None = None) -> str:
    chat_history_for_llm = _build_langchain_history(history or [])

    standalone_question = question
    if chat_history_for_llm:
        try:
            condense_chain = condense_question_prompt | llm
            standalone_question = condense_chain.invoke({
                "question": question,
                "chat_history": chat_history_for_llm,
            }).content
        except Exception as e:
            print(f"[condense] Error: {e}")

    try:
        retrieved_docs = db.similarity_search(standalone_question, k=4)
    except NameError:
        retrieved_docs = vector_store.similarity_search(standalone_question, k=4) # type: ignore

    docs_content = "\n\n".join(
        doc.page_content for doc in retrieved_docs if hasattr(doc, "page_content")
    )

    messages = chatbot_prompt.format_messages(
        question=standalone_question,
        context=docs_content,
    )

    formatted_payload = []
    for msg in messages:
        role = "user" if msg.type == "human" else "system" if msg.type == "system" else "assistant"
        formatted_payload.append({"role": role, "content": msg.content})

    return llm.invoke(formatted_payload).content # type: ignore


# ── TTS / Transcription ───────────────────────────────────────────────────────

def transcribe_audio(audio_path: str) -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY environment variable not set."
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(model="whisper-1", file=audio_file) # type: ignore
    return transcript.text


async def _tts(text: str) -> str:
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural") # type: ignore
    path = "response.mp3"
    await communicate.save(path)
    return path


# ── Text predict (with image generation) ─────────────────────────────────────

# ── Helper: numpy array → base64 data URI ────────────────────────────────────

def _numpy_to_base64(img: np.ndarray) -> str:
    """Convert a numpy HWC uint8 image to a PNG base64 data URI."""
    from PIL import Image
    import io
    pil_img = Image.fromarray(img.astype(np.uint8))
    buffer = io.BytesIO()
    pil_img.save(buffer, format="PNG")
    b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"


# ── Text predict (with image generation) ─────────────────────────────────────

def predict_for_multimodal(
    message: str,
    image_input_numpy: np.ndarray | None,
    history: list,
) -> tuple[list, str, np.ndarray | None]:
    if history is None:
        history = []

    reply = ""
    updated_image_input = None

    try:
        system_content = (
            "You are a financial analyst. "
            "If the user explicitly asks to generate or create an image, call the `generate_dalle_image` tool. "
            "Otherwise, answer their question directly. "
            "If an image is provided, analyze it in the context of the financial question."
        )
        langchain_msgs = [SystemMessage(content=system_content)]
        langchain_msgs.extend(_build_langchain_history(history))

        # ✅ Build multimodal user message if image is present
        if image_input_numpy is not None:
            img_b64_uri = _numpy_to_base64(image_input_numpy)
            # Extract just the base64 data (strip the data URI prefix)
            img_b64_data = img_b64_uri.split(",", 1)[1]

            user_content = [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{img_b64_data}",
                        "detail": "auto",
                    },
                },
                {"type": "text", "text": message or "What do you see in this image?"},
            ]
            langchain_msgs.append(HumanMessage(content=user_content))
        else:
            img_b64_uri = None
            langchain_msgs.append(HumanMessage(content=message))

        llm_with_tools = llm.bind_tools(tools)
        ai_response = llm_with_tools.invoke(langchain_msgs)
        tool_calls = getattr(ai_response, "tool_calls", [])

        if tool_calls:
            tool_call = tool_calls[0]
            name = tool_call["name"] if isinstance(tool_call, dict) else tool_call.name
            args = tool_call["args"] if isinstance(tool_call, dict) else tool_call.args

            if name == "generate_dalle_image":
                visual_prompt = args.get("visual_prompt")
                if visual_prompt:
                    try:
                        url = generate_dalle_image(visual_prompt)
                        if url and url.startswith("data:image"):
                            reply = f'<img src="{url}" style="max-width:100%; border-radius:8px;"/>'
                        elif url:
                            reply = f"![Generated Image]({url})"
                        else:
                            reply = "Sorry, image generation failed. No image data was returned."
                    except Exception as dalle_error:
                        reply = f"Sorry, image generation failed: {dalle_error}"
                else:
                    reply = "I couldn't extract an image prompt from your request."
            else:
                reply = ask_question(message, history)
        else:
            reply = ai_response.content if ai_response.content else ask_question(message, history)

    except Exception as e:
        print(f"[predict_for_multimodal] Error: {e}")
        reply = ask_question(message, history)
        img_b64_uri = None

    # Build the user bubble: show image thumbnail + text if image was provided
    if img_b64_uri:
        user_bubble = f'<img src="{img_b64_uri}" style="max-width:260px; border-radius:8px; display:block; margin-bottom:6px;"/>{message}'
    else:
        user_bubble = message

    history = history + [
        {"role": "user",      "content": user_bubble},
        {"role": "assistant", "content": reply},
    ]
    return history, "", updated_image_input


# ── Voice handler ─────────────────────────────────────────────────────────────

def voice_chat_handler(audio_path: str, history: list) -> tuple[list, str | None, str]:
    if history is None:
        history = []

    if audio_path is None:
        return history, None, "No audio received. Please record your question."

    try:
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            history = history + [
                {"role": "user",      "content": "[Transcription Error]"},
                {"role": "assistant", "content": user_text},
            ]
            return history, None, user_text

        updated_history, _, _ = predict_for_multimodal(user_text, None, history)

        # ✅ Dict format — use ["content"] not [1]
        bot_reply = updated_history[-1]["content"] if updated_history else ""

        if bot_reply and not bot_reply.startswith("<img") and not bot_reply.startswith("!["):
            audio_output_path = asyncio.run(_tts(bot_reply))
            status = "Response generated successfully!"
        else:
            audio_output_path = None
            status = "Image generated successfully!"

        return updated_history, audio_output_path, status

    except Exception as e:
        error = f"Error in voice_chat_handler: {e}"
        print(error)
        history = history + [
            {"role": "user",      "content": "[voice input]"},
            {"role": "assistant", "content": error},
        ]
        return history, None, error


# ── Gradio UI ─────────────────────────────────────────────────────────────────

with gr.Blocks() as multimodal_app:
    gr.Markdown("## Multimodal RAG Assistant")

    # ✅ type="messages" is the critical fix — enforces dict format throughout
    chatbot = gr.Chatbot(label="Chat History", sanitize_html=False)
    status_box = gr.Textbox(label="Voice Status", interactive=False, visible=True)
    audio_out = gr.Audio(label="Voice Response", autoplay=True)

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type a question or ask me to generate an image...",
            scale=4,
        )
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1)

    # ✅ image_input is outside the Row so it renders full-width below
    image_input = gr.Image(
        sources=["upload", "webcam"],
        type="numpy",
        label="📷 Upload Image or Use Camera",
        height=220,
    )
    submit_btn = gr.Button("Submit Text/Image")

    submit_btn.click(
        predict_for_multimodal,
        inputs=[msg, image_input, chatbot],
        outputs=[chatbot, msg, image_input],
    )
    msg.submit(
        predict_for_multimodal,
        inputs=[msg, image_input, chatbot],
        outputs=[chatbot, msg, image_input],
    )
    audio_in.stop_recording(
        voice_chat_handler,
        inputs=[audio_in, chatbot],
        outputs=[chatbot, audio_out, status_box],
    )

multimodal_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://54a999c1f4e830f868.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[Trace(trace_id=tr-e3787f650dbc9f22b4f52444f2171471), Trace(trace_id=tr-5f7ef43b0e4276572272fc90e13e1202)]

## So as usual I downloaded MLFlow cryptography==48.0.0, then i ran the cell again and did my embedding, splitting, and storing in aa vector db, when i performed FAISS, I noticed i got a message saying MLFlow is logging, well it said Expland MLflow trace which is indiciative of logging after running the cell calling autolog (I also renamed thee name to Multimodal RAG chatbot). I then ran my gradio cell and saw the same message and began to prompt. MLFlow trace showed the amount of token i spent, cost break down and how much openAI is billing me to run my chatbot peer prompt, it was less than half a cent.